# SeaDronesSee — FP32 PyTorch GPU vs INT8 TensorRT Hybrid Benchmark

Notebook này benchmark trực tiếp trên Kaggle GPU với bộ SeaDronesSee:

- **FP32**: chạy detector PyTorch CUDA từ checkpoint `fp32_best.pt`.
- **INT8 TensorRT hybrid**: export backbone ConvNeXt từ checkpoint QAT sang ONNX Q/DQ, build TensorRT INT8 engine, sau đó chạy backbone bằng TensorRT còn FPN/RPN/RoI/NMS vẫn chạy PyTorch CUDA.

Checkpoint được lấy từ Kaggle Dataset:

`nguyenducthangtb/echteai-seadronessee-m3-checkpoints`

Lưu ý: TensorRT chỉ dùng ở bước export/build/benchmark, không dùng để train.


In [ ]:
# Cell 1 - Thiết lập tham số chính
from pathlib import Path

REPO_URL = 'https://github.com/NguyenDucThang-tb/EchteAI.git'
REPO = Path('/kaggle/working/EchteAI')
WORK = Path('/kaggle/working/seadronessee_tensorrt_benchmark')
OUTPUT = WORK / 'checkpoints'
LOGS = WORK / 'logs'
OUTPUT.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)

SEADRONESSEE_DATASET = 'ubiratanfilho/sds-dataset'
CHECKPOINT_DATASET = 'nguyenducthangtb/echteai-seadronessee-m3-checkpoints'
RESULT_DATASET = 'nguyenducthangtb/echteai-seadronessee-tensorrt-benchmark'

# None nghĩa là benchmark toàn bộ tập split bên dưới, không giới hạn 100/1000 ảnh nữa.
# Nếu muốn test nhanh, đổi thành 100 hoặc 1000.
BENCHMARK_IMAGES = None
# Dùng split test. Nếu dataset không có annotation test, Cell 5 sẽ fallback test -> val để vẫn tính được mAP.
BENCHMARK_SPLIT = 'test'
BENCHMARK_TAG = 'all' if BENCHMARK_IMAGES is None else str(BENCHMARK_IMAGES)

# Shape cố định cho TensorRT backbone. Với SeaDronesSee, cấu hình cũ thường dùng 960 x 1600.
TENSORRT_HEIGHT = 960
TENSORRT_WIDTH = 1600
TENSORRT_BATCH_SIZE = 1
TENSORRT_WORKSPACE_MB = 4096

# Nếu Kaggle runtime chưa có TensorRT, bật True rồi chạy cell setup, sau đó restart runtime nếu notebook yêu cầu.
INSTALL_TENSORRT_IF_MISSING = False

print('Work dir:', WORK)
print('Benchmark split:', BENCHMARK_SPLIT)
print('Benchmark images:', 'all' if BENCHMARK_IMAGES is None else BENCHMARK_IMAGES)
print('TensorRT shape:', (TENSORRT_BATCH_SIZE, 3, TENSORRT_HEIGHT, TENSORRT_WIDTH))


In [ ]:
# Cell 2 - Clone/pull repo và cài dependency
import os
import subprocess
import sys
from pathlib import Path

if not Path('/kaggle/working').exists():
    raise RuntimeError('Notebook này cần chạy trong Kaggle runtime có /kaggle/working')

if REPO.exists():
    print(f'Repo already exists: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    print(f'Cloning repo from {REPO_URL} -> {REPO}')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True, cwd='/kaggle/working')

os.chdir(REPO)

packages = ['-e', '.[coco]', 'kagglehub', 'onnx', 'onnxscript']
if INSTALL_TENSORRT_IF_MISSING:
    packages.append('tensorrt')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

import kagglehub
import torch
import yaml

sys.path.insert(0, str(REPO))
print('Repository:', Path.cwd())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for idx in range(torch.cuda.device_count()):
    print(f'GPU {idx}:', torch.cuda.get_device_name(idx))


In [ ]:
# Cell 3 - Hàm log chạy subprocess
import datetime
import os
import subprocess
import sys
from pathlib import Path


def run_and_log(command, log_path, cwd=None):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env.setdefault('CUDA_MODULE_LOADING', 'LAZY')
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

    print('Command:', ' '.join(str(x) for x in command), flush=True)
    print('Persistent log:', log_path, flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f'===== START {datetime.datetime.now().isoformat()} =====\n')
        log_file.write('Command: ' + ' '.join(str(x) for x in command) + '\n')
        log_file.flush()
        process = subprocess.Popen(
            command,
            cwd=str(cwd) if cwd else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
        code = process.wait()
        log_file.write(f'===== END code={code} {datetime.datetime.now().isoformat()} =====\n')
    if code != 0:
        raise subprocess.CalledProcessError(code, command)


In [ ]:
# Cell 4 - Tải SeaDronesSee và checkpoint từ Kaggle Dataset M3
import re
from pathlib import Path

import kagglehub
import torch


def find_dataset_root(start):
    start = Path(start)
    candidates = [start]
    candidates += [p.parent.parent for p in start.rglob('instances_train.json')]
    for candidate in candidates:
        if (candidate / 'annotations/instances_train.json').exists() and (candidate / 'images/val').is_dir():
            return candidate
    raise FileNotFoundError(f'Không tìm thấy SeaDronesSee COCO layout trong {start}')


def checkpoint_epoch(path):
    path = Path(path)
    if not path.exists():
        return 0
    payload = torch.load(path, map_location='cpu', weights_only=False)
    if isinstance(payload, dict):
        return int(payload.get('epoch', 0) or 0)
    return 0


def find_checkpoint(root, names):
    root = Path(root)
    matches = []
    for name in names:
        matches.extend(root.rglob(name))
    if not matches:
        return None
    scored = []
    for path in matches:
        try:
            scored.append((checkpoint_epoch(path), str(path), path))
        except Exception:
            scored.append((0, str(path), path))
    scored.sort()
    return scored[-1][2]

DATA_ROOT = find_dataset_root(kagglehub.dataset_download(SEADRONESSEE_DATASET))
CKPT_ROOT = Path(kagglehub.dataset_download(CHECKPOINT_DATASET, force_download=True))

FP32_CHECKPOINT = find_checkpoint(CKPT_ROOT, ['fp32_best.pt', 'fp32_last.pt'])
QAT_CHECKPOINT = find_checkpoint(CKPT_ROOT, ['qat_best.pt', 'qat_last.pt', 'qat_epoch_*.pt'])
SELECTIVE_INT8_CHECKPOINT = find_checkpoint(CKPT_ROOT, ['selective_int8.pt', 'selective_int8_*.pt'])

assert FP32_CHECKPOINT is not None, f'Không tìm thấy fp32_best.pt/fp32_last.pt trong {CKPT_ROOT}'
assert QAT_CHECKPOINT is not None, (
    f'Không tìm thấy QAT checkpoint trong {CKPT_ROOT}. '
    'Để build TensorRT INT8 Q/DQ cần qat_best.pt/qat_last.pt/qat_epoch_*.pt, '
    'checkpoint selective_int8.pt đã convert CPU không đủ để export TensorRT Q/DQ.'
)

print('DATA_ROOT:', DATA_ROOT)
print('CKPT_ROOT:', CKPT_ROOT)
print('FP32 checkpoint:', FP32_CHECKPOINT, 'epoch=', checkpoint_epoch(FP32_CHECKPOINT))
print('QAT checkpoint:', QAT_CHECKPOINT, 'epoch=', checkpoint_epoch(QAT_CHECKPOINT))
print('Selective INT8 CPU checkpoint if present:', SELECTIVE_INT8_CHECKPOINT)


In [ ]:
# Cell 5 - Tạo runtime config cho SeaDronesSee TensorRT benchmark
import yaml
from pathlib import Path

config = yaml.safe_load(Path('configs/seadronessee_colab.yaml').read_text())
config['device'] = 'cuda'

# Ưu tiên test thật nếu có đủ ảnh + annotation. Nếu bản Kaggle không có label test,
# fallback test -> val để vẫn benchmark được mAP trên toàn bộ split có annotation.
TEST_IMAGES = DATA_ROOT / 'images/test'
TEST_ANNOTATIONS = DATA_ROOT / 'annotations/instances_test.json'
if not TEST_IMAGES.is_dir() or not TEST_ANNOTATIONS.exists():
    print('Không thấy test annotation đầy đủ; dùng val làm test split có nhãn để tính mAP.')
    TEST_IMAGES = DATA_ROOT / 'images/val'
    TEST_ANNOTATIONS = DATA_ROOT / 'annotations/instances_val.json'

config['dataset'].update({
    'train_images': str(DATA_ROOT / 'images/train'),
    'train_annotations': str(DATA_ROOT / 'annotations/instances_train.json'),
    'val_images': str(DATA_ROOT / 'images/val'),
    'val_annotations': str(DATA_ROOT / 'annotations/instances_val.json'),
    'test_images': str(TEST_IMAGES),
    'test_annotations': str(TEST_ANNOTATIONS),
    'workers': 2,
})
print('Benchmark test_images:', TEST_IMAGES)
print('Benchmark test_annotations:', TEST_ANNOTATIONS)

# Không tải pretrained backbone khi build model vì checkpoint đã có đủ weight.
config['model']['pretrained_backbone'] = False
config['model']['min_size'] = TENSORRT_HEIGHT
config['model']['max_size'] = TENSORRT_WIDTH
config['model']['anchor_sizes'] = 'auto'
config['model']['anchor_statistics_min_size'] = TENSORRT_HEIGHT

config.setdefault('quantization', {})
config['quantization']['variant'] = 'M3'
config['quantization']['backend'] = 'auto'
config['quantization']['compiler'] = {
    'scope': 'backbone_fpn_rpn_m3',
    'example_batch_size': TENSORRT_BATCH_SIZE,
    'example_height': TENSORRT_HEIGHT,
    'example_width': TENSORRT_WIDTH,
    'artifact_dir': str(OUTPUT / 'tensorrt_artifacts'),
}

config['output'] = {
    'directory': str(OUTPUT),
    'fp32_best': str(FP32_CHECKPOINT),
    'fp32_last': str(FP32_CHECKPOINT),
    'qat_best': str(QAT_CHECKPOINT),
    'qat_last': str(QAT_CHECKPOINT),
    'int8_model': str(SELECTIVE_INT8_CHECKPOINT or OUTPUT / 'selective_int8.pt'),
    'evaluation_json': str(OUTPUT / 'evaluation.json'),
    'benchmark_json': str(OUTPUT / 'benchmark.json'),
    'epoch_benchmarks': str(OUTPUT / 'epoch_benchmarks.json'),
}

RUNTIME_CONFIG = WORK / 'runtime_seadronessee_tensorrt.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Runtime config:', RUNTIME_CONFIG)
print(RUNTIME_CONFIG.read_text())


In [ ]:
# Cell 6 - Kiểm tra TensorRT, export ONNX Q/DQ và build INT8 engine
import importlib.util
from pathlib import Path

import torch

assert torch.cuda.is_available(), 'Hãy bật GPU trong Kaggle Settings trước khi benchmark TensorRT'
if importlib.util.find_spec('tensorrt') is None:
    raise RuntimeError(
        'Môi trường hiện chưa import được TensorRT. '
        'Nếu Kaggle image chưa có TensorRT, bật INSTALL_TENSORRT_IF_MISSING=True ở Cell 1, '
        'chạy lại Cell 2, restart runtime nếu cần rồi chạy tiếp.'
    )

import tensorrt as trt
print('TensorRT:', trt.__version__)

TRT_DIR = OUTPUT / 'tensorrt_artifacts'
TRT_DIR.mkdir(parents=True, exist_ok=True)
INT8_ONNX = TRT_DIR / 'seadronessee_convnext_qat_int8_qdq.onnx'
INT8_ENGINE = TRT_DIR / 'seadronessee_convnext_int8.engine'

common_shape = [
    '--height', str(TENSORRT_HEIGHT),
    '--width', str(TENSORRT_WIDTH),
    '--batch-size', str(TENSORRT_BATCH_SIZE),
]

command = [
    sys.executable, '-u', 'scripts/export_convnext_tensorrt_onnx.py',
    '--config', str(RUNTIME_CONFIG),
    '--model', 'qat_graph',
    '--qat-checkpoint', str(QAT_CHECKPOINT),
    '--output', str(INT8_ONNX),
    '--tensorrt-friendly-int8',
    *common_shape,
]
run_and_log(command, LOGS / 'export_int8_qdq_onnx.log', cwd=REPO)

command = [
    sys.executable, '-u', 'scripts/build_tensorrt_engine.py',
    '--onnx', str(INT8_ONNX),
    '--engine', str(INT8_ENGINE),
    '--precision', 'int8',
    '--workspace-mb', str(TENSORRT_WORKSPACE_MB),
]
run_and_log(command, LOGS / 'build_int8_engine.log', cwd=REPO)

print('INT8 ONNX:', INT8_ONNX, f'{INT8_ONNX.stat().st_size / 2**20:.2f} MB')
print('INT8 TensorRT engine:', INT8_ENGINE, f'{INT8_ENGINE.stat().st_size / 2**20:.2f} MB')


In [ ]:
# Cell 7 - Benchmark FP32 PyTorch GPU vs INT8 TensorRT hybrid GPU trên SeaDronesSee
import json
import time
from pathlib import Path

import torch

from pipelines.convnext_qat.checkpoint import load_checkpoint
from pipelines.convnext_qat.config import load_config, quantized_modules_for_variant
from pipelines.convnext_qat.data import build_coco_loader, unwrap_coco_dataset
from pipelines.convnext_qat.metrics import _coco_metrics, native_detection_metrics
from pipelines.convnext_qat.models import build_fasterrcnn_convnext
from pipelines.convnext_qat.quantization import prepare_selective_qat, set_qat_phase
from scripts.benchmark_convnext_tensorrt_hybrid import TensorRTBackboneRunner, evaluate_hybrid_model

DEVICE = torch.device('cuda')
RESULT_JSON = OUTPUT / f'seadronessee_fp32_pytorch_gpu_vs_int8_tensorrt_m3_hybrid_{BENCHMARK_TAG}.json'

assert FP32_CHECKPOINT.exists(), FP32_CHECKPOINT
assert QAT_CHECKPOINT.exists(), QAT_CHECKPOINT
assert INT8_ENGINE.exists(), INT8_ENGINE

runtime = load_config(str(RUNTIME_CONFIG), require_dataset=True)
loader = build_coco_loader(runtime, BENCHMARK_SPLIT, batch_size=1, shuffle=False, limit=BENCHMARK_IMAGES)
ACTUAL_BENCHMARK_IMAGES = len(loader.dataset)
print('Benchmark split:', BENCHMARK_SPLIT)
print('Benchmark images:', ACTUAL_BENCHMARK_IMAGES, '(full split)' if BENCHMARK_IMAGES is None else f'(limit={BENCHMARK_IMAGES})')

print('Loading FP32 PyTorch checkpoint:', FP32_CHECKPOINT)
fp32_model = build_fasterrcnn_convnext(runtime)
load_checkpoint(FP32_CHECKPOINT, fp32_model, map_location='cpu', strict=True)
fp32_model.to(DEVICE).eval()

print('Loading QAT checkpoint for TensorRT hybrid heads:', QAT_CHECKPOINT)
qat_payload = torch.load(QAT_CHECKPOINT, map_location='cpu', weights_only=False)
metadata = qat_payload.get('extra', {}) if isinstance(qat_payload, dict) else {}
variant = str(metadata.get('variant', runtime['quantization'].get('variant', 'M3'))).upper()
backend = metadata.get('backend', runtime['quantization'].get('backend', 'auto'))
quantized_modules = metadata.get('quantized_modules', quantized_modules_for_variant(runtime, variant))
qat_model = build_fasterrcnn_convnext(runtime)
qat_model = prepare_selective_qat(qat_model, variant, backend, quantized_modules=quantized_modules)
load_checkpoint(QAT_CHECKPOINT, qat_model, map_location='cpu', strict=True)
set_qat_phase(qat_model, 'frozen')
qat_model.to(DEVICE).eval()

print('Loading INT8 TensorRT engine:', INT8_ENGINE)
int8_runner = TensorRTBackboneRunner(INT8_ENGINE)


@torch.inference_mode()
def evaluate_fp32_pytorch_with_timing(model, loader, device, warmup_images=10, progress_frequency=10):
    model.eval()
    predictions, targets, timings = [], [], []
    total_images = len(loader.dataset)
    processed = 0
    print(f'FP32 PyTorch evaluation started: target={total_images} images device={device}', flush=True)

    for images, batch_targets in loader:
        cuda_images = [image.to(device) for image in images]

        torch.cuda.synchronize(device)
        started = time.perf_counter()
        outputs = model(cuda_images)
        torch.cuda.synchronize(device)

        elapsed_ms = (time.perf_counter() - started) * 1000.0 / max(len(images), 1)
        processed += len(images)
        if processed > warmup_images:
            timings.append(elapsed_ms)

        predictions.extend([{key: value.detach().cpu() for key, value in output.items()} for output in outputs])
        targets.extend([
            {key: value.detach().cpu() if torch.is_tensor(value) else value for key, value in target.items()}
            for target in batch_targets
        ])

        if progress_frequency and (processed == 1 or processed % progress_frequency == 0 or processed >= total_images):
            print(f'FP32 progress: {processed}/{total_images} images', flush=True)

    print('FP32 inference completed; calculating metrics', flush=True)
    metrics = native_detection_metrics(predictions, targets)
    dataset = unwrap_coco_dataset(loader.dataset)
    coco = _coco_metrics(predictions, targets, dataset)
    if coco:
        metrics.update(coco)

    avg_ms = sum(timings) / max(len(timings), 1)
    metrics.update({
        'images': int(processed),
        'warmup_images': int(warmup_images),
        'measured_images': max(int(processed) - int(warmup_images), 0),
        'avg_inference_ms_per_image': float(avg_ms),
        'fps': 1000.0 / avg_ms if avg_ms > 0 else None,
        'device': str(device),
        'backend': 'pytorch_cuda',
    })
    return metrics

print('Benchmarking FP32 PyTorch GPU...')
fp32_metrics = evaluate_fp32_pytorch_with_timing(
    fp32_model,
    loader,
    DEVICE,
    warmup_images=min(10, max(ACTUAL_BENCHMARK_IMAGES // 10, 1)),
    progress_frequency=10,
)

print('Benchmarking INT8 TensorRT hybrid GPU...')
int8_metrics = evaluate_hybrid_model(
    qat_model,
    int8_runner,
    loader,
    DEVICE,
    TENSORRT_HEIGHT,
    TENSORRT_WIDTH,
    scope='backbone_fpn_rpn_m3',
    progress_frequency=10,
)
int8_metrics['backend'] = 'tensorrt_int8_hybrid_cuda'

speedup = fp32_metrics['avg_inference_ms_per_image'] / int8_metrics['avg_inference_ms_per_image']
result = {
    'comparison': 'seadronessee_fp32_pytorch_gpu_vs_int8_tensorrt_m3_hybrid_gpu',
    'dataset': SEADRONESSEE_DATASET,
    'checkpoint_dataset': CHECKPOINT_DATASET,
    'fp32_checkpoint': str(FP32_CHECKPOINT),
    'qat_checkpoint': str(QAT_CHECKPOINT),
    'int8_engine': str(INT8_ENGINE),
    'images': int(ACTUAL_BENCHMARK_IMAGES),
    'requested_limit': BENCHMARK_IMAGES,
    'engine_shape': [TENSORRT_BATCH_SIZE, 3, TENSORRT_HEIGHT, TENSORRT_WIDTH],
    'fp32': fp32_metrics,
    'int8_tensorrt_hybrid': int8_metrics,
    'delta_int8_minus_fp32': {
        'map_50_95': int8_metrics.get('map_50_95') - fp32_metrics.get('map_50_95'),
        'map_50': int8_metrics.get('map_50') - fp32_metrics.get('map_50'),
        'latency_ms_per_image': int8_metrics.get('avg_inference_ms_per_image') - fp32_metrics.get('avg_inference_ms_per_image'),
        'fps': int8_metrics.get('fps') - fp32_metrics.get('fps'),
    },
    'speedup': speedup,
}
RESULT_JSON.write_text(json.dumps(result, indent=2, allow_nan=True), encoding='utf-8')

print()
print('FP32 PyTorch GPU:')
print(f"  mAP@50:95: {fp32_metrics.get('map_50_95'):.4f}")
print(f"  mAP@50:    {fp32_metrics.get('map_50'):.4f}")
print(f"  Latency:   {fp32_metrics.get('avg_inference_ms_per_image'):.2f} ms/image")
print(f"  FPS:       {fp32_metrics.get('fps'):.2f}")

print()
print('INT8 TensorRT hybrid GPU:')
print(f"  mAP@50:95: {int8_metrics.get('map_50_95'):.4f}")
print(f"  mAP@50:    {int8_metrics.get('map_50'):.4f}")
print(f"  Latency:   {int8_metrics.get('avg_inference_ms_per_image'):.2f} ms/image")
print(f"  FPS:       {int8_metrics.get('fps'):.2f}")

print()
print('Summary:')
print(f'  Speedup: {speedup:.4f}x')
print('  Saved:', RESULT_JSON)


In [ ]:
# Cell 8 - Vẽ biểu đồ tóm tắt benchmark
import json
from pathlib import Path

import matplotlib.pyplot as plt

assert RESULT_JSON.exists(), RESULT_JSON
result = json.loads(RESULT_JSON.read_text())
fp32 = result['fp32']
int8 = result['int8_tensorrt_hybrid']

labels = ['FP32 PyTorch GPU', 'INT8 TRT hybrid GPU']
latencies = [fp32['avg_inference_ms_per_image'], int8['avg_inference_ms_per_image']]
fps_values = [fp32['fps'], int8['fps']]
map_values = [fp32.get('map_50_95'), int8.get('map_50_95')]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(labels, latencies, color=['tab:blue', 'tab:green'])
axes[0].set_title('Latency thấp hơn là tốt hơn')
axes[0].set_ylabel('ms/image')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(labels, fps_values, color=['tab:blue', 'tab:green'])
axes[1].set_title('FPS cao hơn là tốt hơn')
axes[1].set_ylabel('FPS')
axes[1].tick_params(axis='x', rotation=15)

axes[2].bar(labels, map_values, color=['tab:blue', 'tab:green'])
axes[2].set_title('mAP@50:95')
axes[2].set_ylabel('mAP')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
PLOT_PATH = OUTPUT / f'seadronessee_fp32_vs_int8_tensorrt_{BENCHMARK_TAG}.png'
fig.savefig(PLOT_PATH, dpi=160)
print('Saved plot:', PLOT_PATH)
plt.show()


In [ ]:
# Cell 9 - Upload kết quả benchmark thành Kaggle Dataset
# Cell này giữ lại ONNX, TensorRT engine, log, JSON benchmark và biểu đồ.
import kagglehub
from pathlib import Path

for path in sorted(WORK.rglob('*')):
    if path.is_file():
        print(path.relative_to(WORK), f'{path.stat().st_size / 2**20:.2f} MB')

kagglehub.dataset_upload(
    RESULT_DATASET,
    str(WORK),
    version_notes=f'SeaDronesSee FP32 PyTorch GPU vs INT8 TensorRT hybrid GPU, {BENCHMARK_TAG} split images',
)
print('Uploaded:', f'https://www.kaggle.com/datasets/{RESULT_DATASET}')
